# Artisys Video Studio — primeira cena gratuita

Notebook para **gerar um único MP4 de 5 segundos** no Google Colab gratuito, sem Cloudflare, Modal, API paga ou GPU no computador.

**Motor do piloto:** [CogVideoX-2B](https://huggingface.co/zai-org/CogVideoX-2b), modelo aberto **Apache 2.0**, com difusão text-to-video. Usaremos 49 quadros a 8 FPS (720×480) e recortaremos a saída final em 5 segundos com FFmpeg. Para a GPU gratuita T4, habilitaremos offload sequencial e VAE tiling. **Não gera image-to-video nesta versão e não substitui o Wan 2.2 da futura P2.**

### No iPhone
1. **Ambiente de execução → Alterar tipo → GPU** (T4, se disponível).
2. Rode as células **1 a 5 em ordem**.
3. Na célula **3**, modifique o prompt; na **4**, aguarde a geração; na **5**, baixe o MP4.

**Limitações:** GPU gratuita e sessão não são garantidas. A primeira execução baixa vários gigabytes e pode falhar por falta de memória, espaço, limite de sessão ou rede. Este notebook **não habilita cobrança nem faz execução em segundo plano**.

Fontes: [CogVideoX-2B](https://huggingface.co/zai-org/CogVideoX-2b) · [Diffusers](https://huggingface.co/docs/diffusers/api/pipelines/cogvideox) · [FAQ Colab](https://research.google.com/colaboratory/intl/pt-BR/faq.html).


In [ ]:
# @title 1 — Confirmar GPU gratuita e espaço de armazenamento
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU indisponível: Ambiente de execução > Alterar tipo > GPU. "
        "Se a opção gratuita não estiver disponível, pare aqui."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
disk = shutil.disk_usage("/content").free / 2**30
print(f"GPU: {gpu} | VRAM: {vram:.1f} GiB | Disco livre: {disk:.1f} GiB")

if vram < 10:
    raise RuntimeError("GPU com menos de 10 GiB de VRAM; aguarde outra GPU gratuita.")
if disk < 18:
    raise RuntimeError("Espaço temporário insuficiente: são necessários ao menos 18 GiB livres.")
print("Pronto para instalar dependências.")


In [ ]:
# @title 2 — Instalar bibliotecas open source (sem API)
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.35.2",
    "transformers==4.49.0",
    "accelerate==1.10.1",
    "imageio-ffmpeg>=0.5.1,<1",
    "sentencepiece>=0.2,<1",
    "safetensors>=0.4.5,<1"
], check=True)
print("Bibliotecas instaladas. Se o Colab pedir reinício, reinicie e rode as células na ordem.")


In [ ]:
# @title 3 — Personalizar a cena
prompt = "A friendly anthropomorphic pink brain with large expressive eyes and little arms waves gently at the camera while floating inside a colorful neural universe. High-quality stylized 3D educational cartoon, soft cinematic lighting, smooth movement, consistent character, no letters, no watermark." # @param {type:"string"}
passos = 30 # @param {type:"integer"}
semente = 42 # @param {type:"integer"}
importar_json_do_editor = False # @param {type:"boolean"}
indice_da_cena = 1 # @param {type:"integer"}

if importar_json_do_editor:
    import json
    from google.colab import files
    print("Envie um projeto JSON com a lista 'scenes' e os campos 'visualPrompt' / 'motionPrompt'.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Envie exatamente um JSON.")
    filename, data = next(iter(uploaded.items()))
    if not filename.lower().endswith(".json") or len(data) > 1_000_000:
        raise ValueError("Envie um .json de no máximo 1 MB.")
    project_data = json.loads(data.decode("utf-8"))
    scenes = project_data.get("scenes") if isinstance(project_data, dict) else None
    if not isinstance(scenes, list) or not 1 <= indice_da_cena <= len(scenes):
        raise ValueError("Lista 'scenes' ou número de cena inválido.")
    scene = scenes[indice_da_cena - 1]
    if not isinstance(scene, dict):
        raise ValueError("A cena escolhida precisa ser um objeto.")
    fragments = [scene.get("visualPrompt", ""), scene.get("motionPrompt", "")]
    if not all(isinstance(fragment, str) for fragment in fragments):
        raise ValueError("Os prompts precisam ser textos.")
    prompt = ". ".join(fragment.strip() for fragment in fragments if fragment.strip())

if not isinstance(prompt, str) or not 0 < len(prompt.strip()) <= 2500:
    raise ValueError("Informe um prompt entre 1 e 2500 caracteres.")
if not 10 <= passos <= 50:
    raise ValueError("Use entre 10 e 50 passos (recomendado: 30).")
if not 0 <= semente < 2**32:
    raise ValueError("Semente deve ser um inteiro positivo de 32 bits.")

print("Prompt:", prompt[:250])
print(f"Passos: {passos}, semente: {semente}")
print("CogVideoX-2B é text-to-video. A primeira imagem ainda não é usada como referência.")


In [ ]:
# @title 4 — Gerar MP4 (pode demorar vários minutos, especialmente na T4)
from pathlib import Path
from time import monotonic
import subprocess
import gc
import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

if not torch.cuda.is_available():
    raise RuntimeError("A GPU não está disponível. Rode a célula 1.")

out = Path("/content/artisys-video-studio")
out.mkdir(parents=True, exist_ok=True)
raw = out / "cena_01_original.mp4"
final = out / "cena_01_5s.mp4"
started = monotonic()

# Modelo público FP16 e offload sequencial; evita depender de placa grande.
# Baixar os pesos pode demorar e a RAM padrão do Colab pode não ser suficiente.
print("Carregando CogVideoX-2B (download inicial de vários GB)...")
pipe = CogVideoXPipeline.from_pretrained(
    "zai-org/CogVideoX-2b",
    torch_dtype=torch.float16,
)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("Gerando cena. Não interrompa a sessão do Colab.")
with torch.inference_mode():
    frames = pipe(
        prompt=prompt,
        num_videos_per_prompt=1,
        num_inference_steps=passos,
        num_frames=49,
        guidance_scale=6.0,
        generator=torch.Generator(device="cuda").manual_seed(semente),
    ).frames[0]

export_to_video(frames, str(raw), fps=8)
subprocess.run([
    "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
    "-i", str(raw), "-t", "5", "-an",
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
    str(final),
], check=True)

del frames, pipe
gc.collect()
torch.cuda.empty_cache()
if not final.exists() or final.stat().st_size == 0:
    raise RuntimeError("O MP4 não foi produzido.")
print(f"Arquivo: {final} | {final.stat().st_size/2**20:.1f} MiB")
print(f"Tempo total: {(monotonic()-started)/60:.1f} minutos")


In [ ]:
# @title 5 — Prévia e download do vídeo
from pathlib import Path
from IPython.display import Video, display

path = Path("/content/artisys-video-studio/cena_01_5s.mp4")
if not path.exists():
    raise FileNotFoundError("Rode a célula 4 até o fim.")

display(Video(str(path), embed=True, width=540))
print("No iPhone, se o download não aparecer, abra o painel Arquivos do Colab.")
baixar_agora = False # @param {type:"boolean"}
if baixar_agora:
    from google.colab import files
    files.download(str(path))
